

### Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---

**🔐 Як безпечно зберігати і підвантажувати API-ключі**

API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.



In [1]:
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["OPENAI_API_KEY"]

In [2]:
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,   # обираю низьку температуру, бо це наукова тема, потрібна точність, а не креативність
    openai_api_key=creds["OPENAI_API_KEY"]
)

In [3]:
prompt = (
    "Explain the topic 'Quantum Computing' in simple term"
    "The answer should include: a definition, key advantages, and current research"
    "Keep it short — up to 200 characters."
)

response = chat.invoke(prompt)
print(response.content)

Quantum computing uses quantum bits (qubits) to perform calculations much faster than classical computers. Advantages include solving complex problems quickly. Current research focuses on error correction and practical applications.


### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [4]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["topic"],
    template=(
        "Explain the topic '{topic}' in simple term"
        "The answer should include: a definition, key advantages, and current research"
        "Keep it short — up to 200 characters."
    )
)

In [5]:
chain = prompt_template | chat

In [6]:
topics = [
    "Bayesian methods in machine learning",
    "Transformers in machine learning",
    "Explainable AI"
]

for topic in topics:
    print(f"Тема: {topic}")
    result = chain.invoke({"topic": topic})
    print(result.content)

Тема: Bayesian methods in machine learning
Bayesian methods in machine learning use probability to update beliefs based on new data. Advantages include flexibility and uncertainty quantification. Current research focuses on scalability and deep learning integration.
Тема: Transformers in machine learning
Transformers are models in machine learning that process data sequences, like text, efficiently. They excel in understanding context, enabling better language tasks. Current research focuses on improving efficiency and applications in various fields.
Тема: Explainable AI
Explainable AI (XAI) makes AI decisions understandable to humans. Key advantages include trust, transparency, and improved decision-making. Current research focuses on enhancing interpretability and fairness.




### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [7]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import create_react_agent, AgentExecutor, Tool
from langchain import hub

In [8]:
search = DuckDuckGoSearchRun()

tools = [
    Tool(
        name="Search",
        func=search.invoke,
        description="Useful for searching for information about scientific publications on the internet"
    )
]

prompt = hub.pull("hwchase17/react")
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [9]:
agent = create_react_agent(chat, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, max_iterations=20, verbose=True, handle_parsing_errors=True)

In [10]:
agent_executor.invoke({
    "input": (
        "Search for 'last AI papers' and list 5 results"
        "When done, write your answer after 'Final Answer:'"
    )
})



> Entering new AgentExecutor chain...
I need to search for the latest AI papers to provide a list of five results. 

Action: Search  
Action Input: 'last AI papers'  
19、last in 殿后者 双语例句 1、She rested her last hope on her husband. 她把最后的希望寄托在丈夫身上。 2、I have not heard from him since writing last. 自上次写信以后， 我没有接到过他的信。 3 … last用法中总共有6种词性，2种核心意思: 最后的，持续 当我们学习一般过去时时会用到last， I visited Beijing last week. 我上周去了北京。 此处的last表示上一次，前一次的意思。 May 8, 2024 · last和the last的区别在于它们的用途和上下文中的具体含义。 首先，在一般的语境中，“last”常常用作形容词或副词，表示“最后的”或“最近过去的”。例如，当我们说“I read the last … May 22, 2024 · - "Last but not the least, I want to acknowledge the contribution of our volunteers." 尽管两个短语都传达了相同的信息，但在正式写作中，"last but not least" 更为常见，也更被广泛接受。 … The final / last word in this dictionary is "zoom". 这本词典的最后一个单词是“zoom”。 last (与first相对)，主要指一个系列 (如顺序、位置、时间等)的最后，暗示其后不跟有其他人或东西。 如： 2. He is …I need to refine my search to find specific recent AI papers rather than general information about the term "last." 

Action: Search  
Action Input: '

{'input': "Search for 'last AI papers' and list 5 resultsWhen done, write your answer after 'Final Answer:'",
 'output': '1. "Sparks of AGI" by Microsoft Research - Analyzes an early version of OpenAI’s GPT-4.\n2. "Scaling Laws for Neural Language Models" - Discusses the performance of language models as they scale.\n3. "A Survey on the Use of AI in Drug Discovery" - Reviews advancements in AI applications for drug discovery.\n4. "Ethics of AI: A Comprehensive Review" - Explores ethical considerations in AI development and deployment.\n5. "Advancements in Reinforcement Learning" - Highlights recent breakthroughs in reinforcement learning techniques.'}



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [11]:
from langchain_experimental.utilities import PythonREPL

python_repl = PythonREPL()
python_tool = Tool(
    name="python_interpreter",
    func=python_repl.run,
    description="Use for any calculations, trends, forecasts. Input must be valid Python code that uses print() to output the result."
)

search = DuckDuckGoSearchRun()
search_tool = Tool(
    name="Search",
    func=search.invoke,
    description="Useful for searching current data: weather, inflation, market demand, economic situation."
)

tools = [search_tool, python_tool]

In [12]:
agent = create_react_agent(chat, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True,
)

In [13]:
agent_executor.invoke({
    "input": (
        "We export oranges from Brazil"
        "In 2022 we exported 200 tons, in 2023 — 190 tons, in 2024 — 210 tons, and in 2025 — 220 tons"
        "Estimate how many oranges we will be able to export in 2026, taking into account weather conditions in Brazil and global demand based on the economic situation"
        "If you cannot find data, calculate the trend based on the available figures"
    )
})



> Entering new AgentExecutor chain...


Python REPL can execute arbitrary code. Use with caution.


To estimate the orange exports for 2026, I will first analyze the trend based on the provided export figures from 2022 to 2025. The figures are as follows:

- 2022: 200 tons
- 2023: 190 tons
- 2024: 210 tons
- 2025: 220 tons

I will calculate the average growth rate over these years and use it to project the export figure for 2026. 

Action: python_interpreter  
Action Input: 
```python
# Given export figures
exports = [200, 190, 210, 220]

# Calculate the growth rates
growth_rates = [(exports[i] - exports[i-1]) / exports[i-1] for i in range(1, len(exports))]

# Calculate the average growth rate
average_growth_rate = sum(growth_rates) / len(growth_rates)

# Estimate the export for 2026 based on the last year's export and the average growth rate
last_export = exports[-1]
estimated_export_2026 = last_export * (1 + average_growth_rate)
print(estimated_export_2026)
```  NameError("name 'exports' is not defined")It seems there was an error in the code execution due to a naming issue. I will

{'input': 'We export oranges from BrazilIn 2022 we exported 200 tons, in 2023 — 190 tons, in 2024 — 210 tons, and in 2025 — 220 tonsEstimate how many oranges we will be able to export in 2026, taking into account weather conditions in Brazil and global demand based on the economic situationIf you cannot find data, calculate the trend based on the available figures',
 'output': 'The estimated orange exports for Brazil in 2026 will be approximately 227.54 tons.'}

Агент не шукав погодні умови у Бразилії і не аналізував економічну ситуацію а також ричі отримував NameError на один і той самий код

Що можна покращити:

переписати опис Search Tool конкретніше, щоб агент розумів, що має спочатку шукати дані, а потім рахувати

додати пам'ять